# Rural Bengali Translation Pipeline
### Run each cell in order. Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1 — Install dependencies
!pip install transformers sentencepiece torch pandas openpyxl sacrebleu -q
!pip install indic-transliteration -q

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
drive.mount("/content/drive", force_remount=True)

In [ ]:
# Cell 3 — Upload your files (corpus_aligned.txt and BanglaRegionalTextCorpus-tnQRMn.xlsx)
# Skip this if you already put them in Google Drive
from google.colab import files
uploaded = files.upload()

In [ ]:
# Cell 4 — Imports and config
import torch
import random
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    AutoModelForSeq2SeqLM, AutoTokenizer,
)
from torch.optim import AdamW

MAX_LEN    = 128
BATCH_SIZE = 8
EPOCHS     = 10
LR         = 2e-5
PATIENCE   = 3
DRIVE_PATH = '/content/drive/MyDrive/bengali_translation'

import os
os.makedirs(DRIVE_PATH, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Cell 5 — Load datasets
def load_pairs(filepath):
    pairs=[]
    with open(filepath,'r',encoding='utf-8') as f:
        for line in f:
            if '|||' in line:
                src,tgt=line.strip().split('|||',1)
                pairs.append((src.strip(),tgt.strip()))
    return pairs

# rural -> standard pairs
r2s_pairs = load_pairs('corpus_aligned.txt')
random.shuffle(r2s_pairs)
r2s_split  = int(0.9*len(r2s_pairs))
r2s_train  = r2s_pairs[:r2s_split]
r2s_val    = r2s_pairs[r2s_split:]

# standard -> english pairs
# load romanized standard bengali paired with english
df = pd.read_excel('BanglaRegionalTextCorpus-tnQRMn.xlsx')
df = df[['Standard_Bangla_Texts','English_Translation']].dropna()
s2e_pairs = list(zip(df['Standard_Bangla_Texts'].tolist(), df['English_Translation'].tolist()))
random.shuffle(s2e_pairs)
s2e_split  = int(0.9*len(s2e_pairs))
s2e_train  = s2e_pairs[:s2e_split]
s2e_val    = s2e_pairs[s2e_split:]

print(s2e_pairs)

print(f'R2S  — Train: {len(r2s_train)} | Val: {len(r2s_val)}')
print(f'S2E  — Train: {len(s2e_train)} | Val: {len(s2e_val)}')

In [ ]:
# Cell 6 — Dataset class and train function
class TranslationDataset(Dataset):
    def __init__(self,pairs,tokenizer,model_type='mbart',src_lang=None,tgt_lang=None,prefix=''):
        self.pairs=pairs
        self.tokenizer=tokenizer
        self.model_type=model_type
        self.src_lang=src_lang
        self.tgt_lang=tgt_lang
        self.prefix=prefix

    def __len__(self): return len(self.pairs)

    def __getitem__(self,idx):
        src,tgt=self.pairs[idx]
        src=self.prefix+str(src)
        if self.model_type=='mbart':
            self.tokenizer.src_lang=self.src_lang
            model_inputs=self.tokenizer(src,max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
            self.tokenizer.src_lang=self.tgt_lang
            labels=self.tokenizer(str(tgt),max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
            self.tokenizer.src_lang=self.src_lang
        else:
            model_inputs=self.tokenizer(src,max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
            labels=self.tokenizer(str(tgt),max_length=MAX_LEN,truncation=True,padding='max_length',return_tensors='pt')
        model_inputs['labels']=labels['input_ids']
        return {k:v.squeeze(0) for k,v in model_inputs.items()}


def train_model(model, train_loader, val_loader, save_path, accum_steps=4):
    optimizer=AdamW(model.parameters(),lr=LR)
    best_val_loss=float('inf')
    no_improve=0
    os.makedirs(save_path,exist_ok=True)
    for epoch in range(1,EPOCHS+1):
        model.train()
        total_loss=0
        optimizer.zero_grad()
        for i,batch in enumerate(train_loader):
            batch={k:v.to(device) for k,v in batch.items()}
            loss=model(**batch).loss / accum_steps
            loss.backward()
            if (i+1)%accum_steps==0:
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
                optimizer.zero_grad()
            total_loss+=loss.item()*accum_steps
            if i%10==0:
                print(f'  Epoch {epoch} | Batch {i}/{len(train_loader)} | Loss: {loss.item()*accum_steps:.4f}')
            if i%100==0 and i>0:
              model.eval()
              val_loss=0
              with torch.no_grad():
                  for batch in val_loader:
                      batch={k:v.to(device) for k,v in batch.items()}
                      val_loss+=model(**batch).loss.item()
              print(f'  --> Val loss at batch {i}: {val_loss/len(val_loader):.4f}')
              model.train()
        model.eval()
        val_loss=0
        with torch.no_grad():
            for batch in val_loader:
                batch={k:v.to(device) for k,v in batch.items()}
                val_loss+=model(**batch).loss.item()
        avg_val=val_loss/len(val_loader)
        print(f'Epoch {epoch} | Train: {total_loss/len(train_loader):.4f} | Val: {avg_val:.4f}')
        if avg_val<best_val_loss:
            best_val_loss=avg_val
            no_improve=0
            model.save_pretrained(save_path)
            print(f'  --> Best saved! (val: {best_val_loss:.4f})\n')
        else:
            no_improve+=1
            print(f'  --> No improvement ({no_improve}/{PATIENCE})\n')
            if no_improve>=PATIENCE:
                print(f'Early stopping at epoch {epoch}')
                break

In [ ]:
from torch.cuda.amp import autocast, GradScaler

def train_model_amp(model, train_loader, val_loader, save_path, accum_steps=8):
    optimizer=AdamW(model.parameters(),lr=LR)
    scaler=GradScaler()
    best_val_loss=float('inf')
    no_improve=0
    os.makedirs(save_path,exist_ok=True)
    for epoch in range(1,EPOCHS+1):
        model.train()
        total_loss=0
        optimizer.zero_grad()
        for i,batch in enumerate(train_loader):
            batch={k:v.to(device) for k,v in batch.items()}
            with autocast():
                loss=model(**batch).loss / accum_steps
            scaler.scale(loss).backward()
            if (i+1)%accum_steps==0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            total_loss+=loss.item()*accum_steps
            if i%10==0:
                print(f'  Epoch {epoch} | Batch {i}/{len(train_loader)} | Loss: {loss.item()*accum_steps:.4f}')
            if i%100==0 and i>0:
                model.eval()
                val_loss=0
                with torch.no_grad():
                    for vbatch in val_loader:
                        vbatch={k:v.to(device) for k,v in vbatch.items()}
                        with autocast():
                            val_loss+=model(**vbatch).loss.item()
                print(f'  --> Val loss at batch {i}: {val_loss/len(val_loader):.4f}')
                model.train()
        model.eval()
        val_loss=0
        with torch.no_grad():
            for vbatch in val_loader:
                vbatch={k:v.to(device) for k,v in vbatch.items()}
                with autocast():
                    val_loss+=model(**vbatch).loss.item()
        avg_val=val_loss/len(val_loader)
        print(f'Epoch {epoch} | Train: {total_loss/len(train_loader):.4f} | Val: {avg_val:.4f}')
        if avg_val<best_val_loss:
            best_val_loss=avg_val
            no_improve=0
            model.save_pretrained(save_path)
            print(f'  --> Best saved! (val: {best_val_loss:.4f})\n')
        else:
            no_improve+=1
            print(f'  --> No improvement ({no_improve}/{PATIENCE})\n')
            if no_improve>=PATIENCE:
                print(f'Early stopping at epoch {epoch}')
                break

## Rural → Standard Bengali

In [ ]:
# Cell 7 — mBART rural -> standard
import gc

print('=== mBART-50: Rural -> Standard ===')
gc.collect()
torch.cuda.empty_cache()
mbart_tok   = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')
mbart_model = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50-many-to-many-mmt').to(device)
train_loader = DataLoader(TranslationDataset(r2s_train,mbart_tok,'mbart','en_XX','en_XX'),batch_size=1,shuffle=True)
val_loader   = DataLoader(TranslationDataset(r2s_val,  mbart_tok,'mbart','en_XX','en_XX'),batch_size=1)
train_model_amp(mbart_model,train_loader,val_loader,f'{DRIVE_PATH}/mbart_r2s',accum_steps=8)
del mbart_model; torch.cuda.empty_cache()

## Standard Bengali → English

In [ ]:
# Cell 10 — mBART standard -> english
print('=== mBART-50: Standard -> English ===')
mbart_tok   = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')
mbart_model = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50-many-to-many-mmt').to(device)
from transformers import GenerationConfig
mbart_model.generation_config.forced_bos_token_id    = mbart_tok.lang_code_to_id['en_XX']
mbart_model.generation_config.decoder_start_token_id = mbart_tok.lang_code_to_id['en_XX']
train_loader = DataLoader(TranslationDataset(s2e_train,mbart_tok,'mbart','bn_XX','en_XX'),batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(TranslationDataset(s2e_val,  mbart_tok,'mbart','bn_XX','en_XX'),batch_size=BATCH_SIZE)
train_model(mbart_model,train_loader,val_loader,f'{DRIVE_PATH}/mbart_s2e')
del mbart_model; torch.cuda.empty_cache()

## Done! All models saved to Google Drive under `bengali_translation/`

In [ ]:
# Refresh Memory
import gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

In [ ]:
# Save Tokens During Ind. Batches

saved_tok   = MBart50TokenizerFast.from_pretrained(f'{DRIVE_PATH}/mbart_r2s')
saved_model = MBartForConditionalGeneration.from_pretrained(f'{DRIVE_PATH}/mbart_r2s').to(device)
saved_model.eval()
print('Loaded!')

In [ ]:
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration

s2e_tok = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')

def translate_s2e(text):
    s2e_tok.src_lang='bn_IN'
    inputs=s2e_tok(str(text),return_tensors='pt',max_length=128,truncation=True).to(device)
    generated=s2e_model.generate(
        **inputs,
        forced_bos_token_id=s2e_tok.lang_code_to_id['en_XX'],
        max_new_tokens=128,
        num_beams=4,
    )
    return s2e_tok.batch_decode(generated,skip_special_tokens=True)[0]

for src,tgt in s2e_val[:5]:
    print(f'  Bengali:  {src}')
    print(f'  Expected: {tgt}')
    print(f'  Got:      {translate_s2e(str(src))}\n')

for src,tgt in s2e_val[:1]:
    print(f'  Bengali:  বিড়ালটি দ্রুত।')
    print(f'  Expected: I need help with this thing.')
    print(f'  Got:      {translate_s2e(str('বিড়ালটি দ্রুত।'))}\n')

In [ ]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def deromanize(text):
    if not isinstance(text,str): return ""
    return transliterate(text,sanscript.ITRANS,sanscript.BENGALI)

def full_pipeline(rural_text):
    standard_rom = translate_r2s(rural_text)      # romanized rural -> romanized standard
    standard_bn  = deromanize(standard_rom)        # romanized standard -> Bengali script
    english      = translate_s2e(standard_bn)      # Bengali script -> English
    return english

In [ ]:
from sacrebleu.metrics import BLEU

bleu = BLEU()

# collect predictions and references
predictions = []
references  = []

for src,tgt in s2e_val:
    pred = translate_s2e(str(src))
    predictions.append(pred)
    references.append(str(tgt))

# calculate BLEU
result = bleu.corpus_score(predictions, [references])
print(f"BLEU score: {result}")